# Rambla SA: Perfiles de riesgo del inversor uruguayo y carteras modelo
**Programación Analítica II** · Proyecto Final (Propuesta C)  
**Docente:** Camila Palomeque


In [1]:
# 1. Ubicarse en la raíz del proyecto y agregarla a sys.path para acceder a src/ y datos/
import os
import sys
from pathlib import Path

# Subir a la raíz si estamos dentro de notebooks/ o reingresar si estamos en el workspace general
if Path.cwd().name == "notebooks":
    os.chdir("..")
elif (Path.cwd() / "Proyecto C").exists():
    os.chdir("Proyecto C")

# Asegurar que la raíz del proyecto esté en sys.path para importar src
raiz_proyecto = str(Path.cwd())
if raiz_proyecto not in sys.path:
    sys.path.insert(0, raiz_proyecto)

# 2. Recarga automática de módulos ante modificaciones en src/
%load_ext autoreload
%autoreload 2

# 3. Dependencias estándar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import Markdown, display

# 4. Módulos propios del proyecto
from src.carga import (
    CARPETA_DATOS,
    ARCHIVOS_DEFAULT,
    verificar_archivo,
    cargar_datos,
    cargar_archivo,
    obtener_tasas_referencia,
)
import src.analisis as an


# Etapa 1 · Análisis exploratorio de datos y correcciones de calidad (20 puntos)


## 1.1 Carga de los archivos declarando tipos y fechas
Cargamos los cuatro archivos de datos de **Rambla SA** declarando tipos y fechas según el diseño:
- `encuesta_inversores.csv`
- `movimientos_cartera.csv` (con `fecha_operacion` parseada como fecha)
- `catalogo_instrumentos.csv`
- `tasas_referencia.csv` (con `fecha` parseada como fecha)


In [2]:
# Carga de los cuatro archivos utilizando la función modular en src/carga.py
dict_datos = cargar_datos(carpeta_datos=CARPETA_DATOS)
tc_uyu, val_ui = obtener_tasas_referencia(CARPETA_DATOS)

# Visualización de las primeras filas de cada dataset
for nombre, df in dict_datos.items():
    an.mostrar_primeras_filas(nombre, df)


### **Archivo: `encuesta_inversores.csv`**

,id_cliente,fecha_respuesta,edad,sexo,departamento,cantidad_dependientes,nivel_educativo,ocupacion,ingreso_mensual_liquido,patrimonio_invertible_usd,...,reaccion_caida_20,objetivo_principal,fuente_informacion,prioridad_plazo_fijo,prioridad_letras_bcu,prioridad_fondos_locales,prioridad_obligaciones,prioridad_etf_exterior,prioridad_inmuebles,prioridad_criptomonedas
0,20504,2026-06-24,52.0,Masculino,Montevideo,0,Terciaria completa,Profesional independiente,141700,4200.0,...,Compraría más,Crecimiento del capital,Amigos y familia,7,6,4,5,1,3,2
1,20466,2026-03-29,62.0,Masculino,Colonia,0,Terciaria completa,Jubilado,$ 131.300,13900.0,...,No haría nada,Generar una renta mensual,Investigación propia,7,6,4,5,2,3,1
2,20432,2026-01-11,40.0,Masculino,Maldonado,0,Terciaria completa,Jubilado,53700,4600.0,...,No haría nada,Complemento jubilatorio,Asesor,7,6,4,3,2,5,1
3,20052,2026-02-21,48.0,Masculino,Canelones,0,Posgrado,Empresario / dueño de negocio,0,27700.0,...,No haría nada,Compra de vivienda,Asesor financiero,7,4,6,5,2,3,1
4,20084,2026-04-01,82.0,Femenino,Canelones,0,Posgrado,Jubilado,85100,12100.0,...,No haría nada,Complemento jubilatorio,Amigos y familia,7,6,3,4,1,5,2


### **Archivo: `movimientos_cartera.csv`**

,id_movimiento,id_cliente,id_instrumento,fecha_operacion,tipo_operacion,moneda,monto
0,700954,20141,4,2023-08-06,Compra,UI,17803.22
1,700912,20135,5,2023-11-28,Suscripción,UYU,46255.68
2,700236,20033,3,26/02/2025,Suscripción,UYU,39224.29
3,700992,20148,8,2023-12-09,Suscripción,UYU,195561.19
4,701697,20255,12,17/05/2024,Compra,USD,4324.05


### **Archivo: `catalogo_instrumentos.csv`**

,id_instrumento,nombre_instrumento,categoria,moneda,inversion_minima,retorno_anual_historico,volatilidad_anual,nivel_riesgo,liquidez
0,1,Plazo fijo en pesos (BROU),Renta fija local,UYU,1000,0.085,0.005,1,Baja
1,2,Plazo fijo en dólares,Renta fija local,USD,500,0.033,0.004,1,Baja
2,3,Letras de Regulación Monetaria (BCU),Renta fija local,UYU,50000,0.092,0.013,1,Media
3,4,Notas del Tesoro en Unidades Indexadas,Renta fija local,UI,30000,0.031,0.026,2,Media
4,5,Fondo de inversión local (AFISA),Fondos,UYU,20000,0.071,0.041,2,Alta


### **Archivo: `tasas_referencia.csv`**

,concepto,valor,unidad,fecha
0,"Inflación anual Uruguay (IPC, últimos 12 meses)",5.40,%,2026-06-30
1,Meta de inflación del BCU (centro del rango),4.50,%,2026-06-30
2,Tasa de política monetaria del BCU,8.75,%,2026-06-30
3,Tipo de cambio peso / dólar,42.80,UYU por USD,2026-06-30
4,Valor de la Unidad Indexada,6.55,UYU por UI,2026-06-30


## 1.2 Dimensiones, tipos de dato y descripción estadística
Reportamos para cada tabla la cantidad de filas y columnas, tipos de datos declarados, cardinalidad de las variables categóricas y el resumen descriptivo de las variables continuas.


In [3]:
for nombre, df in dict_datos.items():
    display(Markdown(f"### **Estructura y estadísticas: `{nombre}`**"))
    an.mostrar_descripcion(df)


### **Estructura y estadísticas: `encuesta_inversores`**

**Dimensiones:** 657 filas × 31 columnas

#### Tipos de dato

,id_cliente,fecha_respuesta,edad,sexo,departamento,cantidad_dependientes,nivel_educativo,ocupacion,ingreso_mensual_liquido,patrimonio_invertible_usd,...,reaccion_caida_20,objetivo_principal,fuente_informacion,prioridad_plazo_fijo,prioridad_letras_bcu,prioridad_fondos_locales,prioridad_obligaciones,prioridad_etf_exterior,prioridad_inmuebles,prioridad_criptomonedas
tipo_dato,int64,str,float64,str,str,int64,str,str,str,float64,...,str,str,str,int64,int64,int64,int64,int64,int64,int64


#### Categorías por columna categórica

...

,concepto,valor,unidad,fecha
tipo_dato,str,float64,str,datetime64[us]


#### Categorías por columna categórica

,cantidad_categorias
concepto,8
unidad,3


#### Resumen de variables numéricas

,count,mean,std,min,25%,50%,75%,max
valor,8.0,10.5,13.242006,2.7,4.4,5.975,8.8625,42.8


## 1.3 Búsqueda de duplicados: formularios vs. operaciones
Diferenciamos dos situaciones conceptualmente distintas:
1. **Formularios enviados dos veces:** encuestas con el mismo `id_cliente` repetido.
2. **Operaciones duplicadas:** movimientos de cartera registrados más de una vez (mismas columnas ignorando `id_movimiento`).


In [4]:
for nombre, df in dict_datos.items():
    display(Markdown(f"### **Duplicados en: `{nombre}`**"))
    an.mostrar_duplicados(df)


### **Duplicados en: `encuesta_inversores`**

**Duplicados exactos:** 17 filas

,id_cliente,fecha_respuesta,edad,sexo,departamento,cantidad_dependientes,nivel_educativo,ocupacion,ingreso_mensual_liquido,patrimonio_invertible_usd,...,reaccion_caida_20,objetivo_principal,fuente_informacion,prioridad_plazo_fijo,prioridad_letras_bcu,prioridad_fondos_locales,prioridad_obligaciones,prioridad_etf_exterior,prioridad_inmuebles,prioridad_criptomonedas
447,20024,2026-02-07,53.0,Masculino,Maldonado,0,Terciaria incompleta,Empleado en relación de dependencia,"22.200,00",3500.0,...,Vendería una parte,Complemento jubilatorio,Redes sociales,7,6,5,4,2,3,1
239,20063,2026-02-01,51.0,Masculino,Río Negro,0,Terciaria completa,Empleado público,$ 90.100,5800.0,...,No haría nada,Crecimiento del capital,Redes sociales,4,6,7,2,3,5,1
647,20077,12/06/2026,52.0,Masculino,Maldonado,0,Terciaria completa,Empleado en relación de dependencia,"106.600,00",14500.0,...,No haría nada,Complemento jubilatorio,Prensa especializada,5,7,2,6,3,4,1
143,20093,2026-03-31,53.0,Masculino,Colonia,1,Terciaria completa,Empleado en relación de dependencia,"89.200,00",58100.0,...,No haría nada,Generar una renta mensual,Prensa especializada,7,5,4,6,1,3,2
252,20125,03/05/2026,53.0,Femenino,Canelones,0,Terciaria completa,Empleado en relación de dependencia,62600.00,10200.0,...,No haría nada,Generar una renta mensual,Redes sociales,6,7,4,3,2,5,1
247,20127,2026-02-03,44.0,Femenino,Maldonado,1,Terciaria completa,Empresario / dueño de negocio,150100,34300.0,...,No haría nada,Crecimiento del capital,Redes Sociales,6,7,4,3,2,5,1
255,20154,24/04/2026,38.0,Masculino,Canelones,1,Secundaria completa,Empleado en relación de dependencia,"30.100,00",7700.0,...,No haría nada,Generar una renta mensual,Investigación propia,6,7,4,3,2,5,1
263,20207,2026-01-03,51.0,Femenino,Montevideo,0,Terciaria incompleta,Empleado en relación de dependencia,"152.200,00",38900.0,...,No haría nada,Crecimiento del capital,Asesor financiero,6,3,4,5,2,7,1
111,20231,2026-04-22,62.0,Femenino,San José,2,Secundaria completa,Empleado en relación de dependencia,"48.500,00",14100.0,...,Compraría más,Preservar el capital,Amigos y familia,7,5,6,3,1,4,2
445,20355,2026-03-03,44.0,Femenino,Montevideo,2,Terciaria completa,Estudiante,24400,2800.0,...,Compraría más,Crecimiento del capital,Banco o corredor de bolsa,6,5,2,4,3,7,1


**Formularios con id_cliente repetido:** 17 de más (34 filas en total)

,id_cliente,fecha_respuesta,edad,sexo,departamento,cantidad_dependientes,nivel_educativo,ocupacion,ingreso_mensual_liquido,patrimonio_invertible_usd,...,reaccion_caida_20,objetivo_principal,fuente_informacion,prioridad_plazo_fijo,prioridad_letras_bcu,prioridad_fondos_locales,prioridad_obligaciones,prioridad_etf_exterior,prioridad_inmuebles,prioridad_criptomonedas
12,20024,2026-02-07,53.0,Masculino,Maldonado,0,Terciaria incompleta,Empleado en relación de dependencia,"22.200,00",3500.0,...,Vendería una parte,Complemento jubilatorio,Redes sociales,7,6,5,4,2,3,1
447,20024,2026-02-07,53.0,Masculino,Maldonado,0,Terciaria incompleta,Empleado en relación de dependencia,"22.200,00",3500.0,...,Vendería una parte,Complemento jubilatorio,Redes sociales,7,6,5,4,2,3,1
239,20063,2026-02-01,51.0,Masculino,Río Negro,0,Terciaria completa,Empleado público,$ 90.100,5800.0,...,No haría nada,Crecimiento del capital,Redes sociales,4,6,7,2,3,5,1
59,20063,2026-02-01,51.0,Masculino,Río Negro,0,Terciaria completa,Empleado público,$ 90.100,5800.0,...,No haría nada,Crecimiento del capital,Redes sociales,4,6,7,2,3,5,1
31,20077,12/06/2026,52.0,Masculino,Maldonado,0,Terciaria completa,Empleado en relación de dependencia,"106.600,00",14500.0,...,No haría nada,Complemento jubilatorio,Prensa especializada,5,7,2,6,3,4,1
647,20077,12/06/2026,52.0,Masculino,Maldonado,0,Terciaria completa,Empleado en relación de dependencia,"106.600,00",14500.0,...,No haría nada,Complemento jubilatorio,Prensa especializada,5,7,2,6,3,4,1
143,20093,2026-03-31,53.0,Masculino,Colonia,1,Terciaria completa,Empleado en relación de dependencia,"89.200,00",58100.0,...,No haría nada,Generar una renta mensual,Prensa especializada,7,5,4,6,1,3,2
106,20093,2026-03-31,53.0,Masculino,Colonia,1,Terciaria completa,Empleado en relación de dependencia,"89.200,00",58100.0,...,No haría nada,Generar una renta mensual,Prensa especializada,7,5,4,6,1,3,2
252,20125,03/05/2026,53.0,Femenino,Canelones,0,Terciaria completa,Empleado en relación de dependencia,62600.00,10200.0,...,No haría nada,Generar una renta mensual,Redes sociales,6,7,4,3,2,5,1
8,20125,03/05/2026,53.0,Femenino,Canelones,0,Terciaria completa,Empleado en relación de dependencia,62600.00,10200.0,...,No haría nada,Generar una renta mensual,Redes sociales,6,7,4,3,2,5,1


...

,id_movimiento,id_cliente,id_instrumento,fecha_operacion,tipo_operacion,moneda,monto
1193,704397,20007,12,2025-09-02,Compra,USD,1733.81
3241,700070,20007,12,2025-09-02,Compra,USD,1733.81
269,700109,20013,14,2025-09-27,Compra,USD,173.64
3494,704429,20013,14,2025-09-27,Compra,USD,173.64
2276,700151,20018,7,2023-06-02,Compra,UI,3114.44
4328,704424,20018,7,2023-06-02,Compra,UI,3114.44
1005,704434,20019,4,2023-05-01,Compra,UI,1414.70
2350,700153,20019,4,2023-05-01,Compra,UI,1414.70
1296,704388,20023,9,2022-10-14,Suscripción,USD,274.53
4337,700186,20023,9,2022-10-14,Suscripción,USD,274.53


### **Duplicados en: `catalogo_instrumentos`**

**Duplicados exactos:** 0 filas

### **Duplicados en: `tasas_referencia`**

**Duplicados exactos:** 0 filas

## 1.4 Cuantificación de valores faltantes por columna
Cuantificamos los valores nulos detectados mediante `isna()` y examinamos una muestra filtrada de las filas afectadas por cada columna.


In [5]:
for nombre, df in dict_datos.items():
    display(Markdown(f"### **Faltantes en: `{nombre}`**"))
    an.mostrar_faltantes(df)


### **Faltantes en: `encuesta_inversores`**

#### Faltantes por columna (`isna()`)

,faltantes,porcentaje
meses_gastos_cubiertos,33,5.02
patrimonio_invertible_usd,29,4.41
objetivo_principal,27,4.11
reaccion_caida_20,20,3.04
ingreso_mensual_liquido,14,2.13
edad,6,0.91


Ejemplo de `meses_gastos_cubiertos` faltante: 33 filas (primeras 3):

,id_cliente,fecha_respuesta,edad,sexo,departamento,cantidad_dependientes,nivel_educativo,ocupacion,ingreso_mensual_liquido,patrimonio_invertible_usd,...,reaccion_caida_20,objetivo_principal,fuente_informacion,prioridad_plazo_fijo,prioridad_letras_bcu,prioridad_fondos_locales,prioridad_obligaciones,prioridad_etf_exterior,prioridad_inmuebles,prioridad_criptomonedas
52,20185,2026-01-09,38.0,Masculino,Maldonado,3,Posgrado,Profesional independiente,S/D,52800.0,...,Vendería todo,Crecimiento del capital,Prensa especializada,6,7,5,3,2,4,1
78,20355,2026-03-03,44.0,Femenino,Montevideo,2,Terciaria completa,Estudiante,24400,2800.0,...,Compraría más,Crecimiento del capital,Banco o corredor de bolsa,6,5,2,4,3,7,1
106,20093,2026-03-31,53.0,Masculino,Colonia,1,Terciaria completa,Empleado en relación de dependencia,"89.200,00",58100.0,...,No haría nada,Generar una renta mensual,Prensa especializada,7,5,4,6,1,3,2


...

#### Faltantes por columna (`isna()`)

No hay valores nulos en esta tabla.

### **Faltantes en: `tasas_referencia`**

#### Faltantes por columna (`isna()`)

No hay valores nulos en esta tabla.

## 1.5 Valores raros, formas encubiertas de «no contestó» y rangos imposibles
- **Tipos y fechas:** Búsqueda de valores no numéricos en columnas cuantitativas y fechas no parseables.
- **No respuestas encubiertas:** La función `isna()` detecta solo una de las múltiples formas en que los encuestados omiten respuestas (ej. `"prefiero no contestar"`, `"s/d"`, `"ns/nc"`, o centinelas como `-999`).
- **Rangos imposibles:** Valores fuera de límites lógicos (edades < 18 o > 100, montos negativos, o años invirtiendo superiores a la edad vivida).


In [6]:
for nombre, df in dict_datos.items():
    display(Markdown(f"### **Validación de consistencia: `{nombre}`**"))
    an.mostrar_valores_raros(df)
    an.mostrar_rangos_imposibles(df)


### **Validación de consistencia: `encuesta_inversores`**

`ingreso_mensual_liquido` tiene **26** valores no numéricos

,frecuencia
ingreso_mensual_liquido,
Prefiero no contestar,18
S/D,8


#### Formas de decir «no contestó»

,nulos_isna,disfrazados_o_centinelas,total_no_contesto
columna,,,
ingreso_mensual_liquido,14,26,40
patrimonio_invertible_usd,29,14,43


...

### **Validación de consistencia: `tasas_referencia`**

#### Formas de decir «no contestó»

No se encontraron respuestas encubiertas.

#### Validación de rangos lógicos

No se encontraron números fuera de rango.

## 1.6 Distinción de campos vacíos: errores de carga vs. preguntas salteadas
Diferenciamos los campos vacíos derivados de errores técnicos del sistema o de carga (que deben recuperarse o aislarse sin imputación arbitraria) de las preguntas salteadas por el cliente en la encuesta (que revelan una decisión de reserva y deben etiquetarse como categoría propia `"No contesta"` sin imputar la media).


In [7]:
for nombre, df in dict_datos.items():
    display(Markdown(f"### **Campos vacíos: `{nombre}`**"))
    an.mostrar_campos_vacios(df)


### **Campos vacíos: `encuesta_inversores`**

#### Clasificación de campos vacíos

,columna,tipo,cantidad,tratamiento
0,edad,salteada por el encuestado,6,Categoría 'No contesta'; no imputar la media
1,ingreso_mensual_liquido,salteada por el encuestado,14,Categoría 'No contesta'; no imputar la media
2,patrimonio_invertible_usd,salteada por el encuestado,29,Categoría 'No contesta'; no imputar la media
3,meses_gastos_cubiertos,salteada por el encuestado,33,Categoría 'No contesta'; no imputar la media
4,reaccion_caida_20,salteada por el encuestado,20,Categoría 'No contesta'; no imputar la media
5,objetivo_principal,salteada por el encuestado,27,Categoría 'No contesta'; no imputar la media


,tipo,total_celdas,tratamiento
0,salteada por el encuestado,129,Categoría 'No contesta'; no imputar la media


### **Campos vacíos: `movimientos_cartera`**

...

#### Clasificación de campos vacíos

No se encontraron campos vacíos.

### **Campos vacíos: `tasas_referencia`**

#### Clasificación de campos vacíos

No se encontraron campos vacíos.

## 1.7 ¿Quién es el que no contesta? Faltantes de ingreso y patrimonio
Investigamos si los faltantes de ingreso y patrimonio ocurren al azar o si siguen un patrón no aleatorio (MNAR). Contrastamos el patrimonio declarado contra la **cartera real custodiada** por la firma en `movimientos_cartera.csv` para descubrir el perfil patrimonial de quienes no responden y fundamentar la decisión metodológica de la etapa.


In [8]:
df_encuesta = dict_datos["encuesta_inversores"]
df_mov = dict_datos["movimientos_cartera"]

# 1. Distribución sociodemográfica de no respuesta en la encuesta
an.validar_faltantes(df_encuesta)

# 2. Contraste contra el patrimonio real transaccionado y justificación de la decisión
an.validar_patrimonio(df_encuesta, df_mov, tc_uyu=tc_uyu, val_ui=val_ui)


### **1.7 Distribución de faltantes: Ingreso y Patrimonio**

,nulos_isna,disfrazados_o_centinela,total_faltantes,pct_faltantes
ingreso_mensual_liquido,14,35,49,7.46
patrimonio_invertible_usd,29,14,43,6.54


#### Faltantes según `ocupacion`

,total_clientes,faltan_ingresos,pct_falta_ingreso,faltan_patrimonio,pct_falta_patrimonio
ocupacion,,,,,
Empresario / dueño de negocio,76,12,15.79,9,11.84
Estudiante,44,3,6.82,5,11.36
Empleado público,96,3,3.12,7,7.29
Profesional independiente,88,10,11.36,6,6.82
Jubilado,98,4,4.08,6,6.12
Empleado en relación de dependencia,255,17,6.67,10,3.92


#### Faltantes según `nivel_educativo`

...

#### Cartera real según si declaró patrimonio

,clientes,media_usd,mediana_usd,minimo_usd,maximo_usd
estado_patrimonio,,,,,
Declaró patrimonio,607,19526.98,12486.03,-5658.61,260735.33
No contestó,43,31078.95,20400.41,577.16,125613.41


#### No respuesta por quintil de cartera

,total_clientes,sin_declarar,pct_sin_declarar,monto_promedio_usd
quintil_cartera_real,,,,
Q1 (Baja),130,7,5.38,2440.24
Q2,130,3,2.31,6863.14
Q3,131,8,6.11,12760.78
Q4,129,12,9.30,21856.14
Q5 (Alta),130,13,10.00,57605.60


> **¿Quién es el que no contesta?**
> Los clientes que no declararon patrimonio tienen una cartera real
> con una mediana de **USD 20,400.41**, que es un **63.4% mayor**
> que la de los que sí declararon (**USD 12,486.03**).
> La no respuesta se concentra en los quintiles más altos (Q4 y Q5).
> Son clientes con patrimonios altos que eligen no contestar por privacidad.
>
> **Decisión para el análisis:**
> - **No borrar las filas:** borraríamos al segmento con más fondos.
> - **No imputar con la media:** subestimaría a los clientes de mayor patrimonio.
> - **Solución:** usamos como patrimonio el dinero que tienen invertido en la firma
>   (`cartera_real_usd`), y guardamos un indicador de que no declararon.